In [1]:
import spacy as sp
nlp = sp.load('en_core_web_sm')

def lower_replace(series):
    series = series.str.lower()
    series = series.str.replace(r"\[.*?\]", "", regex=True)
    series = series.str.replace(r"[^a-z0-9\s]", "", regex=True)
    return series

def important_words(text):
    doc = nlp(text)
    words = []
    for token in doc:
        if not token.is_stop:
            words.append(token.lemma_)
    sentence = ' '.join(words)
    return sentence

def nlp_pipeline(series):
    series = lower_replace(series)
    series = series.apply(important_words)
    return series

In [2]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

In [3]:
df = pd.read_csv('final_data.csv')

In [4]:
#try----------------------------------------------
df = df.drop_duplicates(subset=['Title'])
df = df.reset_index(drop=True)

In [5]:
cv = CountVectorizer(max_features=10000, stop_words='english')
dtm = cv.fit_transform(df['tags'])

from sklearn.metrics.pairwise import cosine_similarity
similarities = cosine_similarity(dtm)

In [6]:
#DTM - Word Count Matrix
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=10000, stop_words = 'english')
dtm = cv.fit_transform(df['tags'])
dtm_df = pd.DataFrame(data = dtm.toarray(),  columns = cv.get_feature_names_out())

In [7]:
# cv.get_feature_names_out()[101:1000]

In [8]:
#Lets calculate cosine similarity - movie relationship in percentage
from sklearn.metrics.pairwise import cosine_similarity
similarities = cosine_similarity(dtm_df)

In [9]:
similarities[0]
df

,Book_ID,Title,tags
0,1,atomic habit,",atomic habit,hachette,clear,self help,james"
1,2,rich dad poor dad,",t,finance,kiyosaki,rich dad poor dad,robert,r..."
2,3,alchemist,",fiction,alchemist,coelho,paulo,random house"
3,4,ikigai,"garca,,hachette,self help,hctor,ikigai"
4,5,deep work,",macmillan,productivity,cal,deep work,newport"
5,6,psychology money,"housel,,psychology money,finance,macmillan,morgan"
6,7,think grow rich,",hill,self help,simon & schuster,think grow ri..."
7,8,harry potter philosopher stone,",harry potter philosopher stone,k,fantasy,j,ro..."
8,9,hobbit,",hachette,jrr,tolkien,fantasy,hobbit"
9,10,kill mockingbird,",harper,lee,kill mockingbird,macmillan,classic"


In [10]:
def get_book_index(Title):
    Title = Title.lower().strip()

    for i in df.index:
        if Title == str(df.loc[i, 'Title']).lower().strip():
            return i

    return -1

In [11]:
def get_book_name(index):
    return df.iloc[index]['Title']

In [12]:
Title = "deep work"

index = get_book_index(Title)

similarity_index = list(enumerate(similarities[index]))
similarity_index = sorted(similarity_index, key=lambda x: x[1], reverse=True)

print(similarity_index[:10])

[(4, np.float64(1.0000000000000002)), (5, np.float64(0.1666666666666667)), (9, np.float64(0.1666666666666667)), (13, np.float64(0.1666666666666667)), (14, np.float64(0.1543033499620919)), (19, np.float64(0.1543033499620919)), (0, np.float64(0.0)), (1, np.float64(0.0)), (2, np.float64(0.0)), (3, np.float64(0.0))]


In [13]:
print(similarity_index[:10])

[(4, np.float64(1.0000000000000002)), (5, np.float64(0.1666666666666667)), (9, np.float64(0.1666666666666667)), (13, np.float64(0.1666666666666667)), (14, np.float64(0.1543033499620919)), (19, np.float64(0.1543033499620919)), (0, np.float64(0.0)), (1, np.float64(0.0)), (2, np.float64(0.0)), (3, np.float64(0.0))]


In [14]:
Title = input("Enter book name you read: ").lower().strip()

index = get_book_index(Title)

if index == -1:
    print("Sorry, book not found")
else:
    similarity_index = list(enumerate(similarities[index]))
    similarity_index = sorted(similarity_index, key=lambda x: x[1], reverse=True)

    print("Predicted next 5 books")

    count = 0
    for book_index, score in similarity_index:
        if book_index != index:      
            print(get_book_name(book_index))
            count += 1

        if count == 5:
            break

Enter book name you read:  deep work


Predicted next 5 books
psychology money
kill mockingbird
kite runner
da vinci code
power habit


In [15]:
pickle.dump(similarity_scores, open("similarities.pkl", "wb"))

NameError: name 'pickle' is not defined